# Exp 12: does one human demonstration help? Demo vs no demo on cuRobo

**One-line claim (to be checked against the data below):** with the same generator, planner and repair budget, an arm that sees one
demonstration (plus a scene description and contact evidence) produces programs that transfer to an unseen object scale,
while a task-text-only arm does not. All numbers in this notebook are computed from the logged `events.jsonl` files of the two runs.

**Design.** Same task (`put the bowl on the plate`, `libero_goal_swap` task 8), same model (`muse-wonder-1.0`), same planner
(`curobo-sim`: cuRobo with the robot mask from the simulator, held bowl attached with its known size), up to 4 repairs, debugging on
seed 51 in the scaled scene (bowl x1.227, plate x1.855). Two samples per arm (temperature 0.2 and 0.7). Evaluation of each final
program on held-out seeds 1 to 5 in the scaled scene (where it was debugged) and in the stock scene (never seen during repair).

| | Demo arm | No-demo arm |
|---|---|---|
| Demonstration (DemoSpec, clip 233859) | yes | no |
| Scene description S4 in every prompt | yes | no |
| Contact evidence E3 in every repair | yes | no |

**The arms differ in three things, not one.** The comparison measures "demo + scene + evidence" against "task text only". Separating the
demonstration from the other two needs a third arm (demo, no scene, no evidence); see the last section.

In [ ]:
import json, math, random
from collections import defaultdict
from pathlib import Path

HERE = Path.cwd()
ROOT = next((p for p in (HERE, HERE / "exp12", HERE / "docs" / "synthesizer" / "exp12") if (p / "demo" / "events.jsonl").exists()), None)
assert ROOT is not None, "open this notebook from docs/synthesizer/exp12 or from the repo root"

def load_events(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

ARMS = ("demo", "nodemo")
EV = {a: load_events(ROOT / a / "events.jsonl") for a in ARMS}
CFG = {a: json.loads((ROOT / a / "config.json").read_text(encoding="utf-8")) for a in ARMS}
for a in ARMS:
    c = CFG[a]
    print(f"{a:7s} planner={c['planner']} attach_known={c['attach_known']} evidence={c['evidence']} scene_text={c['scene_text_file']} "
          f"repairs={c['repairs']} temps={c['temperatures']} eval_seeds={c['eval_seeds']} budget={c['token_budget']}")

In [ ]:
# tidy records
evals = [dict(arm=e["arm"], sample=e["sample"], scene=e["scene"], seed=e["seed"], ok=bool(e["success"]), reason=e["reason"])
         for a in ARMS for e in EV[a] if e["kind"] == "eval"]
temps = {(a, e["sample"]): e["temperature"] for a in ARMS for e in EV[a] if e["kind"] == "sample_start"}
rollouts = [e for a in ARMS for e in EV[a] if e["kind"] == "rollout"]
programs = [e for a in ARMS for e in EV[a] if e["kind"] == "program"]
run_end = {a: next(e for e in EV[a] if e["kind"] == "run_end") for a in ARMS}
SCENES = sorted({r["scene"] for r in evals})
print("scenes:", SCENES, "| eval rows:", len(evals), "| rollouts:", len(rollouts), "| programs:", len(programs))

In [ ]:
# --- small exact statistics, standard library only ---
def _bisect(f, lo=0.0, hi=1.0, it=200):
    flo = f(lo)
    for _ in range(it):
        mid = (lo + hi) / 2
        if (f(mid) > 0) == (flo > 0):
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

def binom_cdf(k, n, p):
    return sum(math.comb(n, i) * p**i * (1 - p) ** (n - i) for i in range(k + 1))

def wilson(k, n, z=1.959964):
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def clopper_pearson(k, n, alpha=0.05):
    lo = 0.0 if k == 0 else _bisect(lambda p: (1 - binom_cdf(k - 1, n, p)) - alpha / 2)
    hi = 1.0 if k == n else _bisect(lambda p: binom_cdf(k, n, p) - alpha / 2)
    return (lo, hi)

def fisher_exact(a, b, c, d):
    # two-sided exact test for [[a, b], [c, d]]: a/b successes/failures of arm 1, c/d of arm 2
    n1, n2, K = a + b, c + d, a + c
    N = n1 + n2
    def pmf(x):
        return math.comb(K, x) * math.comb(N - K, n1 - x) / math.comb(N, n1)
    p_obs = pmf(a)
    return min(1.0, sum(pmf(x) for x in range(max(0, n1 - (N - K)), min(K, n1) + 1) if pmf(x) <= p_obs * (1 + 1e-9)))

def newcombe_diff(k1, n1, k2, n2):
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wilson(k1, n1)
    l2, u2 = wilson(k2, n2)
    d = p1 - p2
    return d, d - math.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), d + math.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)

def pct(x):
    return f"{100 * x:5.1f}%"

## 1. Success rates on held-out seeds

Pooled over the two samples and the five seeds (10 evaluations per cell). The 95% intervals are exact (Clopper-Pearson) and Wilson.
**Caution:** the 10 evaluations are not independent (the same 5 seeds are used by both samples, and each sample's program is one draw), so
these intervals are optimistic. Section 2 repeats the comparison with seeds as the unit.

In [ ]:
rows = []
for scene in SCENES:
    for a in ARMS:
        k = sum(r["ok"] for r in evals if r["arm"] == a and r["scene"] == scene)
        n = sum(1 for r in evals if r["arm"] == a and r["scene"] == scene)
        cp, w = clopper_pearson(k, n), wilson(k, n)
        rows.append((scene, a, k, n, k / n, cp, w))
print(f"{'scene':8s} {'arm':7s} {'k/n':>6s} {'rate':>7s}  {'Clopper-Pearson 95%':>22s}  {'Wilson 95%':>16s}")
for scene, a, k, n, p, cp, w in rows:
    print(f"{scene:8s} {a:7s} {k:>3d}/{n:<2d} {pct(p)}  [{pct(cp[0])}, {pct(cp[1])}]  [{pct(w[0])}, {pct(w[1])}]")

print()
print("per sample (rate on 5 held-out seeds):")
for scene in SCENES:
    for a in ARMS:
        cells_ = []
        for s in sorted({r["sample"] for r in evals if r["arm"] == a}):
            rs = [r["ok"] for r in evals if r["arm"] == a and r["scene"] == scene and r["sample"] == s]
            cells_.append(f"T={temps[(a, s)]}: {sum(rs)}/{len(rs)}")
        print(f"  {scene:8s} {a:7s} " + " | ".join(cells_))

## 2. Arm comparison

Two views: (a) a pooled 2x2 table with Fisher's exact test and a Newcombe interval for the difference, which treats the 10 evaluations
per arm as independent; (b) a **paired analysis with seeds as the unit**, which is the safer one because both arms are evaluated on the same
seeds. For (b), each seed gets a score per arm (the share of that arm's samples that succeeded on that seed), the paired differences are
tested with an exact sign-flip permutation test (all 2^5 flips), and the mean difference gets a bootstrap interval over seeds.

In [ ]:
print("(a) pooled 2x2, demo vs nodemo")
for scene in SCENES:
    k1 = sum(r["ok"] for r in evals if r["arm"] == "demo" and r["scene"] == scene); n1 = sum(1 for r in evals if r["arm"] == "demo" and r["scene"] == scene)
    k2 = sum(r["ok"] for r in evals if r["arm"] == "nodemo" and r["scene"] == scene); n2 = sum(1 for r in evals if r["arm"] == "nodemo" and r["scene"] == scene)
    d, lo, hi = newcombe_diff(k1, n1, k2, n2)
    p = fisher_exact(k1, n1 - k1, k2, n2 - k2)
    print(f"  {scene:7s} demo {k1}/{n1} vs nodemo {k2}/{n2}: diff {100*d:+.0f} pts (Newcombe 95% [{100*lo:+.0f}, {100*hi:+.0f}]), Fisher exact p = {p:.4f}")

print()
print("(b) paired by seed (seed = unit of analysis)")
by = defaultdict(lambda: defaultdict(list))
for r in evals:
    by[(r["arm"], r["scene"])][r["seed"]].append(r["ok"])
rng = random.Random(0)
PAIRED = {}
for scene in SCENES:
    seeds = sorted(by[("demo", scene)])
    sc = {a: {s: sum(by[(a, scene)][s]) / len(by[(a, scene)][s]) for s in seeds} for a in ARMS}
    diffs = [sc["demo"][s] - sc["nodemo"][s] for s in seeds]
    obs = sum(diffs) / len(diffs)
    flips = [sum(sg * d for sg, d in zip(signs, diffs)) / len(diffs)
             for signs in __import__("itertools").product((1, -1), repeat=len(diffs))]
    p_perm = sum(abs(f) >= abs(obs) - 1e-12 for f in flips) / len(flips)
    boots = sorted(sum(rng.choice(diffs) for _ in diffs) / len(diffs) for _ in range(10000))
    lo, hi = boots[249], boots[9749]
    PAIRED[scene] = dict(mean_diff=obs, ci=(lo, hi), p_perm=p_perm, per_seed=dict(zip(seeds, diffs)))
    print(f"  {scene:7s} mean per-seed difference {obs:+.2f} (bootstrap 95% [{lo:+.2f}, {hi:+.2f}]), exact sign-flip p = {p_perm:.4f} "
          f"(minimum attainable with 5 seeds: {2 / 2 ** len(diffs):.4f})")
    print("          per-seed differences:", {s: round(d, 2) for s, d in zip(seeds, diffs)})

## 3. The debugging phase (seed 51, scaled scene)

How many attempts each sample needed, and why the failed attempts failed. A sample that never solved the debug seed is still evaluated
with its last program, so the evaluation numbers above include programs that failed on the debug seed.

In [ ]:
print(f"{'arm':7s} {'sample':>6s} {'T':>4s} {'attempts':>8s} {'solved':>7s}  attempt reasons")
DEBUG = {}
for a in ARMS:
    for s in sorted({r["sample"] for r in rollouts if r["arm"] == a}):
        rs = sorted((r for r in rollouts if r["arm"] == a and r["sample"] == s), key=lambda r: r["attempt"])
        solved = any(r["success"] for r in rs)
        DEBUG[(a, s)] = dict(attempts=len(rs), solved=solved)
        print(f"{a:7s} {s:>6d} {temps[(a, s)]:>4} {len(rs):>8d} {str(solved):>7s}  " + ", ".join(r["reason"] for r in rs))

## 4. Cost

Tokens come from the `program` events (one per generated or repaired program); the run total (`run_end.new_tokens`) is the ledger delta for
the whole command and is the number the budget cap applies to. Dollars use the logged `usd_per_mtok`.

In [ ]:
print(f"{'arm':7s} {'programs':>8s} {'tokens (sum of program events)':>32s} {'of which reasoning':>20s} {'run new_tokens':>15s} {'USD est.':>9s}")
COST = {}
for a in ARMS:
    ps = [p for p in programs if p["arm"] == a]
    tok = sum(p["tokens"] for p in ps)
    rea = sum(p.get("reasoning_tokens") or 0 for p in ps)
    COST[a] = dict(programs=len(ps), tokens=tok, reasoning=rea, new_tokens=run_end[a]["new_tokens"], usd=run_end[a]["usd_est"])
    print(f"{a:7s} {len(ps):>8d} {tok:>32,d} {rea:>20,d} {run_end[a]['new_tokens']:>15,d} {run_end[a]['usd_est']:>9.2f}")
solved = {a: sum(1 for (aa, _), v in DEBUG.items() if aa == a and v["solved"]) for a in ARMS}
print()
for a in ARMS:
    print(f"{a:7s} solved the debug seed in {solved[a]}/2 samples; new tokens per solved sample: "
          + (f"{COST[a]['new_tokens'] / solved[a]:,.0f}" if solved[a] else "n/a (none solved)"))

## 5. Figure

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

labels = [f"{scene}\n{a}" for scene in SCENES for a in ARMS]
vals = [next(p for (sc, aa, k, n, p, cp, w) in rows if sc == scene and aa == a) for scene in SCENES for a in ARMS]
errs = []
for scene in SCENES:
    for a in ARMS:
        k, n = next((k, n) for (sc, aa, k, n, p, cp, w) in rows if sc == scene and aa == a)
        lo, hi = clopper_pearson(k, n)
        errs.append((k / n - lo, hi - k / n))
if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 3.4))
    colors = ["#1f6feb" if "\ndemo" in l else "#8b949e" for l in labels]
    ax.bar(range(len(vals)), vals, color=colors, yerr=[[e[0] for e in errs], [e[1] for e in errs]], capsize=4)
    ax.set_xticks(range(len(vals)), labels)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("held-out success rate (10 evaluations)")
    ax.set_title("Exp 12: final programs on held-out seeds 1 to 5 (error bars: exact 95% CI)")
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib is not installed (the repo has no plotting dependency); text bars instead:\n")
    for l, v, e in zip(labels, vals, errs):
        print(f"{l.replace(chr(10), ' '):16s} {'#' * round(v * 40):<40s} {pct(v)} [-{100*e[0]:.0f}, +{100*e[1]:.0f}]")

## 6. Findings (generated from the numbers above)

In [ ]:
def rate(scene, arm):
    k = sum(r["ok"] for r in evals if r["arm"] == arm and r["scene"] == scene)
    return k, sum(1 for r in evals if r["arm"] == arm and r["scene"] == scene)

for scene in SCENES:
    kd, nd = rate(scene, "demo"); kn, nn = rate(scene, "nodemo")
    P = PAIRED[scene]
    print(f"- {scene} scene: demo {kd}/{nd}, no demo {kn}/{nn}; paired by seed {P['mean_diff']:+.2f} "
          f"(bootstrap 95% [{P['ci'][0]:+.2f}, {P['ci'][1]:+.2f}], exact p = {P['p_perm']:.3f}).")
print(f"- Debug seed solved: demo {solved['demo']}/2 samples, no demo {solved['nodemo']}/2 samples.")
print(f"- New tokens: demo {COST['demo']['new_tokens']:,}, no demo {COST['nodemo']['new_tokens']:,}.")
print()
print("Reading rule: with 5 seeds the exact sign-flip test cannot go below p =", f"{2 / 2 ** 5:.4f}", "(two-sided), so it cannot reach p < 0.05 at all.")
print("Judge the paired effect by its bootstrap interval and size, and treat the pooled Fisher p-value above as optimistic (it counts 10 non-independent evaluations).")
print("A claim at the 0.05 level needs more seeds: the held-out range has 50.")

## 7. Threats to validity and what would settle them

- **Three differences between the arms, not one.** The demo arm also sees the S4 scene description and E3 evidence. Run a third arm (demo only) to isolate the demonstration.
- **Scene and robot mask are privileged.** `curobo-sim` takes the robot mask from the simulator and `--attach-known` gives the planner the true bowl size. Both arms get the same help, but the absolute numbers are not what a real sensor stack would give.
- **Tiny sample.** Two samples per arm, one run each, five seeds. Intervals are wide. Use seeds 1 to 50 and at least five samples per arm for any claim beyond "signal".
- **Non-independence.** Seeds are shared across samples and arms; use the paired analysis (section 2b) or a mixed model, not the pooled test, for inference.
- **The debug scene is the scaled scene.** The scaled-scene evaluation is therefore a within-distribution test and the stock-scene evaluation is the transfer test.
- **Attach frame bug.** Both arms ran before the fix: the held bowl's points were removed from the cloud, but its spheres sat 9.65 cm too high, inside the wrist, so cuRobo did not model the bowl as an obstacle. Attempts could fail or pass for that reason; a rerun on the fixed code is needed before quoting these numbers as the attached-bowl setting.
- **Planner noise.** cuRobo refusals can be false negatives (see `docs/synthesizer/handoff-curobo-debug.md`); a failed rollout may reflect the planner, not the program. Replays of the same program can differ.
- **Acceptance vs held-out.** Debugging used seed 51 (acceptance range 51 to 65). Evaluation used seeds 1 to 5 from the held-out range 1 to 50, which were never used for repair.

## 8. Reproducing

- Settings: `docs/synthesizer/demo-vs-nodemo-config.json`, run book entries `exp12.demo` and `exp12.nodemo`
  (`uv run python -m ants.evalkit.runbook cmd exp12.demo`).
- Raw logs: `data/runs/demo_e3_s4_sim/` and `data/runs/nodemo_task_only_sim/` (git-ignored). The files this notebook reads
  (`events.jsonl`, `config.json`, `summary.md`) are copied next to it in `docs/synthesizer/exp12/`.

In [ ]:
# machine-readable summary for the dashboard / write-up
summary = dict(
    rates={f"{scene}/{a}": dict(k=k, n=n, rate=p, clopper_pearson=cp, wilson=w) for (scene, a, k, n, p, cp, w) in rows},
    paired={s: dict(mean_diff=v["mean_diff"], bootstrap_ci=v["ci"], p_exact=v["p_perm"]) for s, v in PAIRED.items()},
    debug={f"{a}/{s}": v for (a, s), v in DEBUG.items()},
    cost=COST,
)
out = ROOT / "results_summary.json"
out.write_text(json.dumps(summary, indent=2, default=list) + "\n", encoding="utf-8")
print("wrote", out)